# Cars vs Noncars Dataset Inspection

Use this notebook to inspect the prepared dataset and verify split/class balance.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

DATASET_DIR = Path('./datasets/cars_noncars_daynight')
DATASET_DIR.resolve()

In [ ]:
records = []
for split in ['day', 'night']:
    for klass in ['cars', 'noncars']:
        folder = DATASET_DIR / split / klass
        if not folder.exists():
            continue
        for p in folder.rglob('*'):
            if p.is_file() and p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff'}:
                records.append({
                    'split': split,
                    'class': klass,
                    'filename': p.name,
                    'path': str(p),
                    'size_bytes': p.stat().st_size,
                })

df = pd.DataFrame(records)
print('rows:', len(df))
df.head(10)

In [ ]:
if len(df) > 0:
    display(df.groupby(['split', 'class']).size().rename('count').reset_index())
    display(df.groupby('class')['size_bytes'].describe())

In [ ]:
sample_n = 6
sample_df = df.sample(min(sample_n, len(df)), random_state=42) if len(df) else df
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.flatten()
for ax in axes:
    ax.axis('off')

for ax, (_, row) in zip(axes, sample_df.iterrows()):
    img = Image.open(row['path']).convert('RGB')
    ax.imshow(img)
    ax.set_title(f"{row['split']} | {row['class']}", fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()